# Movies and Audience Ratings
**Daniel Yarritu | BAN 6003**

Milestone 1 profiles the five supplied files. Milestone 2 builds and validates a preliminary analytic base table (ABT) with one row per movie and checks the ratings aggregation in DuckDB.

## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)

# Milestone 3 and final project examples are disabled.
RUN_FUTURE_MILESTONES = False


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3).iloc[:, :6])  # bounded preview; full schema appears below


movies: 9,082 rows x 21 columns
ratings: 99,810 rows x 5 columns
links: 9,125 rows x 3 columns
credits: 9,082 rows x 3 columns
keywords: 9,082 rows x 2 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language
0,1,862,tt0114709,Toy Story,Toy Story,en
1,2,8844,tt0113497,Jumanji,Jumanji,en
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

Out[0]: 
      table   rows  columns  duplicate_rows  missing_cells
0    movies   9082       21               0             72
1   ratings  99810        5               0              0
2     links   9125        3               0             13
3   credits   9082        3               0            114
4  keywords   9082        2               0            761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


Out[0]: 
      table      expected_key  duplicate_keys  missing_key_values
0    movies           movieId               0                   0
1    movies            tmdbId               0                   0
2   ratings  userId + movieId               0                   0
3     links           movieId               0                   0
4   credits            tmdbId               0                   0
5  keywords            tmdbId               0                   0


## Milestone 1: Initial Profile and Cleaning Plan

The following sections explain the project question, data sources, initial findings, cleaning plan, possible outcome, and responsible-use concerns.


### Scope of this milestone

I used the Canvas Milestone 1 checklist to organize this work. This stage focuses on understanding the data and planning the cleaning steps. Integration and modeling will follow in later milestones.


### Business problem and project question
A streaming service could use audience ratings to help review movies for its catalog. This project will explore whether movie characteristics are related to average audience ratings.

**How well can genre, runtime, release year, and original language predict average MovieLens ratings for movies with at least 20 ratings?**

The intended user is a streaming service catalog analyst. The results could help identify types of movies worth reviewing, but ratings alone would not determine which titles to acquire. Licensing costs and the service's audience would also matter.

### Data sources and table descriptions
I am using the five CSV files supplied in the professor's [Option 1 template](https://github.com/tianhaiz/ban6003-project-option-1-movies-template). The [data guide](https://github.com/tianhaiz/ban6003-project-option-1-movies-template/blob/main/data/README.md) explains the files and their identifiers. The starter identifies the data as a curated subset of The Movies Dataset and MovieLens ratings. All findings here refer to these supplied files.

| Raw file | What one row represents | Expected key | Planned use |
|---|---|---|---|
| movies_metadata.csv | One movie | movieId; also unique tmdbId | Main table with genre, runtime, year, and language |
| ratings.csv | One user's rating for one movie | userId + movieId | Calculate average ratings and rating counts per movie |
| links.csv | One movie's MovieLens, IMDb, and TMDB identifier mapping | movieId | Check that the movie identifiers agree |
| credits_curated.csv | One movie's director and main cast | tmdbId | Add director and cast information if useful |
| keywords_curated.csv | One movie's keyword text | tmdbId | Add content descriptions if useful |

The code checks table sizes, data types, missing values, duplicate records and keys, unusual values, and matches between tables. It also saves file checksums so the original inputs can be identified. The raw CSV files are unchanged.

### Possible outcome and final table
My proposed outcome is the **average MovieLens rating for each movie**, on a 0.5-5 scale. Following the starter's guidance, I plan to include movies with at least 20 ratings in the initial model. This reduces the influence of averages based on only a few people, but it does not guarantee reliable ratings. I will review other thresholds later.

MovieLens ratings and TMDB vote_average are different measures. TMDB uses a 0-10 scale, so I will not substitute it for the MovieLens outcome. Rating count will help decide which movies have enough data; it will not be an input to the initial model.

The final ABT will have one row per movie, identified by movieId. I will summarize ratings by movieId before joining them to the movie table, then use tmdbId to connect credits and keywords. After each join, I will check row counts, duplicate keys, and unmatched movies. Titles are not suitable keys because different movies can have the same title.

### Initial cleaning plan
1. I will clean copies of the data and record what changes and how many rows are affected.
2. I will check identifiers before converting links.tmdbId to a nullable integer type. Missing identifiers will stay missing. IMDb identifiers will be handled as strings when building the identifier mapping.
3. I will convert release_date to a date and check it against release_year. I will also compare rating_datetime with the Unix timestamp. The initial checks found no date or timestamp mismatches.
4. I will investigate zero budgets and revenues instead of assuming they represent actual amounts. These fields are not needed for my initial model. I will treat zero runtime as missing and review very long runtimes before deciding whether to exclude them.
5. I will trim extra spaces and review missing categories. For missing genres or other descriptions, I will use an unknown category where appropriate rather than invent information. I will split pipe-separated lists when needed and use ast.literal_eval to safely read nested list text. I will preserve multiple genres when preparing genre inputs.
6. I will review any duplicates before removing them. For missing model inputs, I will document the handling method. If I fill missing values using estimates, I will calculate those estimates from the training data only.
7. For the later analysis of released movies, I will initially use titles marked Released and document exclusions. I will review other status values against release dates and recalculate the number of eligible movies after filtering.
8. I will summarize valid ratings and check the final movie-level table. The initial features will exclude rating summaries, rating dates, TMDB voting measures, popularity, and observed revenue to avoid using information closely tied to the outcome.

### Ethics and limitations
User IDs should not be treated as fully anonymous. I will report movie-level summaries and will not try to identify individual users. The ratings come from people who chose to rate movies, so they may not represent all viewers. Requiring 20 ratings also favors movies with more coverage. I will check whether missing information and rating coverage differ by language or release period.

These are historical data, so the results may not reflect current audience preferences. Ratings describe reported opinions, not objective movie quality. Any relationships found will not establish cause and effect. The project uses the supplied data for this class; broader reuse would require checking the source terms.


In [6]:
# Full column-level profile (before cleaning)
column_profile = pd.concat([
    pd.DataFrame({
        'table': name, 'column': df.columns,
        'dtype': [str(t) for t in df.dtypes],
        'missing_count': df.isna().sum().values,
        'missing_percent': (100 * df.isna().mean()).round(2).values,
        'distinct_nonmissing': df.nunique().values,
    }) for name, df in tables.items()
], ignore_index=True)
display(column_profile)
column_profile.to_csv(OUTPUT_DIR / 'milestone1_column_profile.csv', index=False)


,table,column,dtype,missing_count,missing_percent,distinct_nonmissing
0,movies,movieId,int64,0,0.00,9082
1,movies,tmdbId,int64,0,0.00,9082
2,movies,imdb_id,object,0,0.00,9082
3,movies,title,object,0,0.00,8809
4,movies,original_title,object,0,0.00,8841
5,movies,original_language,object,0,0.00,42
6,movies,release_date,object,0,0.00,6002
7,movies,runtime,float64,0,0.00,235
8,movies,budget,int64,0,0.00,599
9,movies,revenue,float64,0,0.00,4555


In [7]:
# Suspicious values and timestamp/date consistency
release_dates = pd.to_datetime(movies['release_date'], errors='coerce')
rating_dates = pd.to_datetime(ratings['rating_datetime'], errors='coerce', utc=True)
unix_dates = pd.to_datetime(ratings['timestamp'], unit='s', errors='coerce', utc=True)
checks = {
    'budget equals zero (review unknown-like values)': movies['budget'].eq(0).sum(),
    'revenue equals zero (review unknown-like values)': movies['revenue'].eq(0).sum(),
    'runtime equals zero': movies['runtime'].eq(0).sum(),
    'runtime exceeds 300 minutes (review, not automatic error)': movies['runtime'].gt(300).sum(),
    'negative budget, revenue, or runtime': movies[['budget','revenue','runtime']].lt(0).any(axis=1).sum(),
    'invalid release dates': release_dates.isna().sum(),
    'release year differs from parsed date': (release_dates.notna() & release_dates.dt.year.ne(movies['release_year'])).sum(),
    'ratings outside 0.5-5 or half-point increments': (~ratings['rating'].between(0.5,5) | ratings['rating'].mul(2).mod(1).ne(0)).sum(),
    'invalid rating dates or timestamps': (rating_dates.isna() | unix_dates.isna()).sum(),
    'rating datetime differs from Unix timestamp': (rating_dates.notna() & unix_dates.notna() & rating_dates.ne(unix_dates)).sum(),
    'TMDB vote_average outside 0-10': (~movies['vote_average'].between(0,10)).sum(),
}
suspicious_checks = pd.Series(checks, name='affected_rows').to_frame()
display(suspicious_checks)
display(movies.loc[movies['runtime'].gt(300), ['movieId','title','runtime']])
suspicious_checks.to_csv(OUTPUT_DIR / 'milestone1_value_checks.csv')


,affected_rows
budget equals zero (review unknown-like values),4443
revenue equals zero (review unknown-like values),4322
runtime equals zero,20
"runtime exceeds 300 minutes (review, not automatic error)",15
"negative budget, revenue, or runtime",0
invalid release dates,0
release year differs from parsed date,0
ratings outside 0.5-5 or half-point increments,0
invalid rating dates or timestamps,0
rating datetime differs from Unix timestamp,0


,movieId,title,runtime
4862,6896,Shoah,566.0
5213,7762,Tinker Tailor Soldier Spy,320.0
5216,7767,The Best of Youth,366.0
5870,27002,From the Earth to the Moon,720.0
5881,27251,The 10th Kingdom,417.0
6234,38499,Angels in America,352.0
6899,58649,Napoleon,332.0
7224,69241,Berlin Alexanderplatz,931.0
7767,86487,Mildred Pierce,336.0
7981,93040,The Civil War,680.0


In [8]:
# Coverage and text/category checks; this is profiling, not ABT construction.
coverage = pd.Series({
    'rating events with no matching movie': (~ratings['movieId'].isin(movies['movieId'])).sum(),
    'catalog movies without ratings': (~movies['movieId'].isin(ratings['movieId'])).sum(),
    'link rows outside curated movie catalog': (~links['movieId'].isin(movies['movieId'])).sum(),
    'movies without credits match': (~movies['tmdbId'].isin(credits['tmdbId'])).sum(),
    'movies without keywords match': (~movies['tmdbId'].isin(keywords['tmdbId'])).sum(),
}, name='affected_rows')
display(coverage.to_frame())
mapping_check = movies[['movieId','tmdbId']].merge(
    links[['movieId','tmdbId']], on='movieId', how='left',
    suffixes=('_movies','_links'), validate='one_to_one')
display(pd.Series({
    'movie rows missing linked TMDB ID': mapping_check['tmdbId_links'].isna().sum(),
    'conflicting nonmissing TMDB mappings': (mapping_check['tmdbId_links'].notna() & mapping_check['tmdbId_movies'].ne(mapping_check['tmdbId_links'])).sum(),
}).to_frame('affected_rows'))
text_checks = []
for name, df in tables.items():
    for column in df.select_dtypes(include=['object','string']).columns:
        values = df[column].dropna().astype(str)
        text_checks.append({'table':name, 'column':column,
            'outer_whitespace': int(values.ne(values.str.strip()).sum()),
            'blank_strings': int(values.str.strip().eq('').sum()),
            'empty_list_strings': int(values.str.strip().eq('[]').sum())})
display(pd.DataFrame(text_checks))
display(movies['status'].value_counts(dropna=False).to_frame('movie_count'))
display(movies['primary_genre'].value_counts(dropna=False).to_frame('movie_count'))
rating_counts = ratings.groupby('movieId').size()
display(pd.Series({
    'movies with any ratings': len(rating_counts),
    'movies with at least 20 ratings': rating_counts.ge(20).sum(),
    'rated movies below threshold': rating_counts.lt(20).sum(),
    'median ratings per rated movie': rating_counts.median(),
    'minimum rating': ratings['rating'].min(),
    'maximum rating': ratings['rating'].max(),
    'earliest rating (UTC)': rating_dates.min(),
    'latest rating (UTC)': rating_dates.max(),
}).to_frame('value'))
# Record raw-file versions for reproducibility.
import hashlib, platform
raw_manifest = pd.DataFrame([
    {'file': str(path.relative_to(REPO_ROOT)),
     'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
    for path in sorted(DATA_DIR.glob('*.csv'))
])
raw_manifest.to_csv(OUTPUT_DIR / 'milestone1_raw_file_manifest.csv', index=False)
key_audit.to_csv(OUTPUT_DIR / 'milestone1_key_audit.csv', index=False)
print(f'Python {platform.python_version()}; pandas {pd.__version__}; numpy {np.__version__}')


Python 3.12.14; pandas 2.2.3; numpy 2.3.5


,affected_rows
rating events with no matching movie,0
catalog movies without ratings,57
link rows outside curated movie catalog,43
movies without credits match,0
movies without keywords match,0


,affected_rows
movie rows missing linked TMDB ID,0
conflicting nonmissing TMDB mappings,0


,table,column,outer_whitespace,blank_strings,empty_list_strings
0,movies,imdb_id,0,0,0
1,movies,title,0,0,0
2,movies,original_title,0,0,0
3,movies,original_language,0,0,0
4,movies,release_date,0,0,0
5,movies,genres,0,0,35
6,movies,production_companies,0,0,664
7,movies,production_countries,0,0,294
8,movies,status,0,0,0
9,movies,genres_list,0,0,0


,movie_count
status,
Released,9061
Rumored,11
Post Production,7
NaN,2
In Production,1


,movie_count
primary_genre,
Drama,2413
Comedy,2168
Action,1177
Adventure,517
Horror,497
Crime,417
Documentary,383
Thriller,263
Fantasy,244


,value
movies with any ratings,9025
movies with at least 20 ratings,1300
rated movies below threshold,7725
median ratings per rated movie,3.0
minimum rating,0.5
maximum rating,5.0
earliest rating (UTC),1995-01-09 11:46:49+00:00
latest rating (UTC),2016-10-16 17:57:24+00:00


### Initial findings

| Table | Rows | Columns |
|---|---:|---:|
| Movies | 9,082 | 21 |
| Ratings | 99,810 | 5 |
| Links | 9,125 | 3 |
| Credits | 9,082 | 3 |
| Keywords | 9,082 | 2 |

**Duplicates and identifiers.** I found no full-row duplicates or duplicate values in the expected keys checked. A movieId can appear many times in ratings because different users rate the same movie. The userId-movieId pairs are unique. All rating records match the movie catalog, and the nonmissing TMDB mappings agree between movies and links. Links contains 43 rows outside the curated movie table; these are extra identifier mappings, not duplicate movies.

**Missing information and data types.** Movies has 2 missing status values and 35 missing values each in genres_list and primary_genre. Credits has 24 missing directors and 90 missing cast entries. Keywords are missing for 761 movies. Links has 13 missing tmdbId values, so pandas reads that column as float64 even though it contains identifiers. release_date is read as text and needs date conversion. The nested metadata and pipe-separated lists also need preparation before use as model inputs.

**Zero and unusual values.** There are 4,443 zero budgets, 4,322 zero revenues, and 20 zero runtimes. These values need review rather than being accepted as complete measurements. Fifteen runtimes exceed 300 minutes, including documentaries and series-like titles. A long runtime is not automatically an error. Ratings are all between 0.5 and 5 in half-point steps. Release dates parse successfully, release years agree with those dates, and rating datetimes match their Unix timestamps.

**Text and category issues.** Extra spaces appear in 3 director entries, 3 cast entries, and 18 keyword entries. The nested fields contain 35 empty genre lists, 664 empty production-company lists, and 294 empty production-country lists. These empty lists indicate that descriptive information is absent even when the cell is not marked missing. Status values include 9,061 Released, 11 Rumored, 7 Post Production, 1 In Production, and 2 missing; I will review these before filtering for the later analysis.

**Rating coverage.** There are 57 catalog movies without ratings. Of the 9,025 movies with ratings, only 1,300 have at least 20. The median rated movie has 3 ratings. This means the initial model will use a much smaller group than the full catalog, and high averages based on very few ratings should be interpreted carefully.

**Time coverage.** Rating events span January 9, 1995 through October 16, 2016 (UTC), and movie release years range from 1902 to 2016. This project therefore studies historical audience ratings.

### Next steps
I will apply the cleaning plan, summarize valid ratings by movieId, and build a table with one row per movie. I will then check that the joins preserve the intended rows and document the retained columns. Modeling and final recommendations will be completed in later milestones.


## Milestone 2: Preliminary movie-level ABT

### Updated business question

For movies marked Released with at least 20 MovieLens ratings, how well do genres, runtime, release year, and original language predict the average MovieLens rating? A streaming catalog analyst could use this historical analysis to inform which types of movies to review.

Milestone 1 found a median of three ratings per rated movie and 57 catalog movies with no ratings. The ABT retains the full catalog and flags which movies meet the 20-rating threshold so coverage can be checked before selecting a modeling sample. Release status is recorded separately for the later model filter.

### Clean and transform copies of the input tables

I parsed release dates, derived release year from those dates, and checked it against the supplied year. I normalized language codes to lowercase and trimmed category labels. I trimmed each pipe-separated genre, cast, and keyword value and retained multiple genres in the same movie row. Missing text stays missing. Zero runtimes become missing and unusually long runtimes remain available for review.

I read the links identifiers as nullable integers and strings directly from the CSV to preserve their meaning. I left budget, revenue, popularity, and TMDB voting measures out of the preliminary ABT. The initial model will use the movie attributes in the business question.

In [9]:
movie_columns = [
    "movieId", "tmdbId", "imdb_id", "title", "release_date", "release_year",
    "runtime", "original_language", "status", "genres_list", "primary_genre",
]
movies_clean = movies[movie_columns].copy()
movies_clean["release_date"] = pd.to_datetime(movies_clean["release_date"])
derived_year = movies_clean["release_date"].dt.year.astype("Int64")
assert derived_year.eq(movies_clean["release_year"]).all()
movies_clean["release_year"] = derived_year

zero_runtime = movies_clean["runtime"].eq(0)
movies_clean["runtime"] = movies_clean["runtime"].mask(zero_runtime)
for column in ["original_language", "status", "primary_genre"]:
    movies_clean[column] = movies_clean[column].astype("string").str.strip().replace("", pd.NA)
movies_clean["original_language"] = movies_clean["original_language"].str.lower()

def clean_pipe_list(value):
    return "|".join(part.strip() for part in value.split("|") if part.strip())

movies_clean["genres_list"] = (
    movies_clean["genres_list"].map(clean_pipe_list, na_action="ignore")
    .astype("string").replace("", pd.NA)
)
credits_clean = credits.copy()
credits_clean["director"] = credits_clean["director"].astype("string").str.strip().replace("", pd.NA)
credits_clean["top_cast"] = (
    credits_clean["top_cast"].map(clean_pipe_list, na_action="ignore")
    .astype("string").replace("", pd.NA)
)
keywords_clean = keywords.copy()
keywords_clean["keywords_list"] = (
    keywords_clean["keywords_list"].map(clean_pipe_list, na_action="ignore")
    .astype("string").replace("", pd.NA)
)
links_clean = pd.read_csv(
    DATA_DIR / "links.csv", dtype={"movieId": "Int64", "tmdbId": "Int64", "imdbId": "string"}
).rename(columns={"tmdbId": "linked_tmdbId", "imdbId": "linked_imdb_id"})
links_clean["linked_imdb_id"] = "tt" + links_clean["linked_imdb_id"].str.zfill(7)

print(f"Zero runtimes set to missing: {zero_runtime.sum():,}")
print(f"Runtimes over 300 minutes retained: {movies_clean['runtime'].gt(300).sum():,}")

Zero runtimes set to missing: 20
Runtimes over 300 minutes retained: 15


### Summarize rating events by movie

Each input rating row represents one user's rating of one movie. After checking the user–movie key and rating scale, I group the events by `movieId` to calculate the mean and number of ratings. The resulting table has one row per rated movie. The mean is the potential outcome and the count is used for sample eligibility.

In [10]:
assert ratings[["userId", "movieId"]].notna().all().all()
assert not ratings.duplicated(["userId", "movieId"]).any()
assert ratings["rating"].isin(np.arange(0.5, 5.5, 0.5)).all()
assert ratings["movieId"].isin(movies_clean["movieId"]).all()

rating_summary = (
    ratings.groupby("movieId", as_index=False)
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
    )
)
assert rating_summary["user_rating_count"].sum() == len(ratings)
print(f"{len(ratings):,} rating events summarized into {len(rating_summary):,} movies.")
display(rating_summary.head())

99,810 rating events summarized into 9,025 movies.


,movieId,user_rating_mean,user_rating_count
0,1,3.872470,247
1,2,3.401869,107
2,3,3.161017,59
3,4,2.384615,13
4,5,3.267857,56


### Join and validate at the movie level

One ABT row represents a catalog movie, identified by `movieId`. I use four left joins to keep the catalog as the base. The rating summary and identifier crosswalk join on `movieId`. Credits and keywords already have one row per `tmdbId` and join on that key using the catalog's TMDB identifier. The audit records row counts, duplicate keys, and unmatched rows for each join.

In [11]:
join_checks = []

def join_movie_table(left, right, key, table_name):
    assert left[key].notna().all() and right[key].notna().all()
    joined = left.merge(right, on=key, how="left", validate="one_to_one", indicator=True)
    duplicate_movies = joined["movieId"].duplicated().sum()
    join_checks.append({
        "table": table_name,
        "key": key,
        "rows_before": len(left),
        "right_rows": len(right),
        "rows_after": len(joined),
        "duplicate_movie_ids": duplicate_movies,
        "unmatched_movies": joined["_merge"].eq("left_only").sum(),
        "unused_right_rows": (~right[key].isin(left[key])).sum(),
    })
    assert len(joined) == len(left) and duplicate_movies == 0
    return joined.drop(columns="_merge")

abt = join_movie_table(movies_clean, rating_summary, "movieId", "Rating summary")
abt = join_movie_table(abt, links_clean, "movieId", "Links")
abt = join_movie_table(abt, credits_clean, "tmdbId", "Credits")
abt = join_movie_table(abt, keywords_clean, "tmdbId", "Keywords")
display(pd.DataFrame(join_checks))

assert abt["linked_tmdbId"].notna().all()
assert abt["tmdbId"].eq(abt["linked_tmdbId"]).all()
abt["imdb_id_conflict"] = abt["imdb_id"].ne(abt["linked_imdb_id"])
abt = abt.drop(columns="linked_tmdbId")
abt["user_rating_count"] = abt["user_rating_count"].fillna(0).astype("int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
assert abt["user_rating_mean"].isna().eq(abt["user_rating_count"].eq(0)).all()
assert abt["user_rating_count"].sum() == len(ratings)

missingness = abt.isna().sum().rename("missing_rows").to_frame()
missingness["missing_pct"] = (100 * missingness["missing_rows"] / len(abt)).round(2)
display(missingness)
display(pd.Series({
    "catalog_movies": len(abt),
    "movies_without_ratings": abt["user_rating_count"].eq(0).sum(),
    "movies_with_at_least_20_ratings": abt["rating_target_eligible"].sum(),
    "released_movies_with_at_least_20_ratings": (
        abt["rating_target_eligible"] & abt["status"].eq("Released")
    ).sum(),
    "conflicting_imdb_ids": abt["imdb_id_conflict"].sum(),
}))

,table,key,rows_before,right_rows,rows_after,duplicate_movie_ids,unmatched_movies,unused_right_rows
0,Rating summary,movieId,9082,9025,9082,0,57,0
1,Links,movieId,9082,9125,9082,0,0,43
2,Credits,tmdbId,9082,9082,9082,0,0,0
3,Keywords,tmdbId,9082,9082,9082,0,0,0


,missing_rows,missing_pct
movieId,0,0.00
tmdbId,0,0.00
imdb_id,0,0.00
title,0,0.00
release_date,0,0.00
release_year,0,0.00
runtime,20,0.22
original_language,0,0.00
status,2,0.02
genres_list,35,0.39


catalog_movies                              9082
movies_without_ratings                        57
movies_with_at_least_20_ratings             1300
released_movies_with_at_least_20_ratings    1300
conflicting_imdb_ids                          17
dtype: int64

### DuckDB check of the pandas aggregation

The SQL query independently summarizes the rating events by `movieId`. The comparison checks every movie ID, count, and mean against pandas.

In [12]:
import duckdb

rating_query = """
SELECT
    movieId,
    COUNT(*) AS user_rating_count,
    AVG(rating) AS user_rating_mean
FROM rating_events
GROUP BY movieId
ORDER BY movieId
"""
with duckdb.connect() as connection:
    connection.register("rating_events", ratings)
    sql_rating_summary = connection.sql(rating_query).df()

pd.testing.assert_frame_equal(
    sql_rating_summary, rating_summary[sql_rating_summary.columns],
    check_dtype=False, rtol=1e-10, atol=1e-12,
)
display(sql_rating_summary.head())
print(f"DuckDB and pandas agree for all {len(sql_rating_summary):,} rated movies.")

DuckDB and pandas agree for all 9,025 rated movies.


,movieId,user_rating_count,user_rating_mean
0,1,247,3.872470
1,2,107,3.401869
2,3,59,3.161017
3,4,13,2.384615
4,5,56,3.267857


DuckDB reproduced the counts and means for all 9,025 rated movies. Each join retained 9,082 catalog rows with zero duplicate movie IDs. The 57 movies without ratings have a count of zero and a missing mean.

### Remaining data issues

- The links file contains 43 mappings outside this catalog. TMDB identifiers agree for the joined movies, but 17 IMDb identifiers differ across the two sources. Both IMDb values and a conflict flag are retained for review. The joins use MovieLens and TMDB identifiers.
- Some runtimes, genres, cast entries, directors, and keywords remain missing. The long runtimes and non-Released or missing statuses need review before modeling.
- The 20-rating cutoff leaves 1,300 movies. All 1,300 are marked Released. The resulting sample may favor more frequently rated titles. These ratings are historical and reflect MovieLens users who chose to rate each movie.
- The later model will use genres, runtime, release year, and original language. Rating counts and means are available for eligibility and the outcome. They are excluded from the predictor set to avoid leakage.

In [13]:
abt_path = OUTPUT_DIR / "movies_preliminary_abt.csv"
abt.to_csv(abt_path, index=False)
print(f"Saved {len(abt):,} movie rows to outputs/{abt_path.name}.")

Saved 9,082 movie rows to outputs/movies_preliminary_abt.csv.


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [14]:
if RUN_FUTURE_MILESTONES:
    # Milestone 3 code workspace
    # Suggested sequence:
    # 1. State the prediction or analysis timing in markdown.
    # 2. Filter to rows where your target is defined.
    # 3. Select only features available at that time.
    # 4. Check class balance or the target distribution.
    # 5. Create a reproducible train/test split when modeling.
    # 6. Fit a simple baseline and evaluate it on test data.
    pass

In [16]:
if RUN_FUTURE_MILESTONES:
    # Example placeholder. Replace with your chosen target and feature set.
    # target = 'your_target_column'
    # features = ['feature_1', 'feature_2']
    # model_df = abt[[target] + features].dropna()
    
    abt.describe(include='all').T.head(25)
    pass

## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [17]:
if RUN_FUTURE_MILESTONES:
    # Final project code workspace
    # Save final datasets, figures, or tables here when they are ready.
    # Recommended filenames:
    # final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
    # dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
    # metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
    # report_path = REPORTS_DIR / "movies_final_report.html"
    pass

In [19]:
if RUN_FUTURE_MILESTONES:
    # Save final deliverables only after your ABT and documentation are complete.
    # abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
    # data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)
    
    # Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
    pass

## AI Use Disclosure

I used ChatGPT for code and explanations in Milestone 1 and to help draft the Milestone 2 notebook updates, including data cleaning, the movie-level aggregation and joins, the DuckDB validation query, and the accompanying explanations. ChatGPT also ran the notebook and checked the resulting ABT and validation outputs.